# Step 4b — Three-Step GBDT: Gender Gap ↔ Novelty Mechanism

*Female Editors-in-Chief and the Leading-Author Gender Gap — replication code.*

**Input:** `output/derived/2-{AUTHOR}-journal-year-novelty.dta`, created by `04a_novelty_prepare_journal_year.do`. Run that Stata script first.  
**Output:** `output/novelty/{AUTHOR}/` (CSV tables and PDF figures).

**Research question:** how much of a treated journal's post-treatment novelty change is
consistent with its observed gender-gap change, and how much is unexplained by that channel?

Rather than forecasting each outcome independently, this notebook explicitly models
the **gender-gap → novelty mechanism** with three linked models, then decomposes the observed
post-treatment novelty deviation into a gender-gap-implied part and a residual part.

---

## Design

**Unit of analysis:** journal-year.

**Untreated-regime training sample (`did_treat == 0`):** all years of never-treated journals
+ pre-treatment years of treated journals. All three models below are fit on this sample only.

**Model A** — untreated gender-gap trajectory:
`gender_gap = f(L1..L5 gender_gap, year, category, field, controls, gap_pretreat_mean)`.
Novelty is deliberately excluded from Model A's features — the gender gap is assumed not to be
influenced by the level of research novelty, so only the gap's own dynamics, journal controls,
and `gap_pretreat_mean` (a static per-journal feature: the journal's own mean gender gap over
its untreated regime — all years for never-treated, pre-treatment years only for treated — a
fixed-effect-style proxy for the journal's typical gap level) enter this model.

**Model B** — untreated novelty trajectory, gender-gap channel excluded:
`novelty = f(L1..L5 novelty, year, category, field, controls)`

**Model C** — novelty trajectory with the gender-gap channel included:
`novelty = f(gender_gap, L1..L5 gender_gap, L1..L5 novelty, year, category, field, controls)`

Because `novelty` here can be `avg_novel_gap`, `avg_novel_female`, or `avg_novel_male`, the full
A/B/C triplet is estimated **separately for each of the three novelty measures** — 9 GBDT
models in total. `avg_novel_gap` = `avg_novel_male` − `avg_novel_female` (the male-led minus
female-led novel-paper share, i.e. a journal-year's novelty gender gap). Model A's specification does not actually vary across the three
runs (it never sees novelty), so its three fits are conceptually identical — refit
independently per run purely to keep the pipeline uniform across novelty measures, not because
the underlying model differs.

## Recursive post-treatment counterfactual paths

For each treated journal, forecasts are generated year-by-year for the post-treatment period.
The first post-treatment year uses actual pre-treatment lagged values; every later year replaces
post-treatment lagged inputs with **previously predicted** counterfactual values, so no path
leaks actual post-treatment outcomes into itself.

- `pred_gender_gap_0` — Model A, recursive, no-treatment gender-gap path.
- `pred_novelty_B_0` — Model B, recursive, baseline no-treatment novelty path (gap-blind).
- `pred_novelty_C_G0` — Model C, current & lagged gender-gap set to the **predicted**
  no-treatment gender-gap path (`pred_gender_gap_0`); lagged novelty recursively set to this
  same path's own previous predictions.
- `pred_novelty_C_Gactual` — Model C, current & lagged gender-gap set to the **observed**
  post-treatment gender-gap path; lagged novelty recursively set to this same path's own
  previous predictions (never actual post-treatment novelty).

## Decomposition

- `gender_gap_deviation = actual_gender_gap − pred_gender_gap_0`
- `total_novelty_deviation = actual_novelty − pred_novelty_C_G0`
- `gap_implied_novelty_component = pred_novelty_C_Gactual − pred_novelty_C_G0`
- `residual_novelty_component = actual_novelty − pred_novelty_C_Gactual`

`total_novelty_deviation ≈ gap_implied_novelty_component + residual_novelty_component`
(exact except for the small gap between `pred_novelty_C_Gactual` and what a model trained
only on untreated data would predict given the fully actual post-treatment path).

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor  
import sklearn
print(f'sklearn {sklearn.__version__}')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('All imports OK')

In [ ]:
# ─── Leading-author definition ───────────────────────────────────────────────
AUTHOR = 'first'  # 'first' or 'last' — run the notebook once for each

# ─── Paths (relative to this notebook, which lives in code/analysis/) ────────
ROOT     = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # replication package root
DATA_DIR = os.path.join(ROOT, 'output', 'derived')                   # created by 04a_*.do
FIG_DIR  = os.path.join(ROOT, 'output', 'novelty', AUTHOR)
os.makedirs(FIG_DIR, exist_ok=True)


# ─── Gender gap (Model A/C target / feature) ──────────────────────────────────
GAP_COL = 'gender_gap_y'

# ─── Novelty measures — each gets its own Model A/B/C triplet ────────────────
# 'avg_novel' (all-papers novelty) is replaced with 'avg_novel_gap', the male-minus-female
# novelty-share gap (avg_novel_male - avg_novel_female), computed below right after load.
NOVELTY_VARS = {
    'avg_novel_gap'    : 'Novelty gap (male - female share novel)',
    'avg_novel_female' : 'Novelty (female-led papers)',
    'avg_novel_male'   : 'Novelty (male-led papers)',
}

# ─── Journal-level controls (shared across all models) ───────────────────────
CONTROLS = [
    'n_papers', 'avg_num_authors',
    'log_jrn_avgcited', 'log_jrn_totref', 'log_jrn_refperdoc',
    'jrn_hindex', 'jrn_sjr', 'avg_top_inst'
]
CATEGORICAL_CONTROLS = ['field', 'category']

N_LAGS = 3   # number of lagged values used as features

# Minimum papers a journal-year must have to enter model *training*. gender_gap_y and
# avg_novel* are averages/ratios computed over that journal-year's papers, so low-n_papers
# rows are noisier outcome measurements, not more informative ones -- including them mostly
# adds label noise for the GBDT to fit. Restricting training only (lags/features are still
# built from the full, unfiltered panel, so the year-to-year lag alignment is untouched)
# drops the noisiest tail of untreated-regime rows; counterfactual generation for treated
# journals still uses every actual observation regardless of n_papers.
MIN_PAPERS = 10

# ─── Panel identifiers ────────────────────────────────────────────────────────
JID = 'journal_id'
YR  = 'Publication_Year'

print('Config OK')
print(f'Output will be saved to: output/novelty/{AUTHOR}')
print(f'Novelty measures: {list(NOVELTY_VARS)}')
print(f'Controls: {CONTROLS}')
print(f'MIN_PAPERS (training-sample restriction): {MIN_PAPERS}')

In [ ]:
# ─── Load journal-year level data ─────────────────────────────────────────────
path = os.path.join(DATA_DIR, f'2-{AUTHOR}-journal-year-novelty.dta')
df = pd.read_stata(path, convert_categoricals=False)

# ─── Type cleaning ────────────────────────────────────────────────────────────
for col in [YR, 'ever_treated', 'did_treat', 'post']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

df['first_treat_yr'] = pd.to_numeric(df['first_treat_yr'], errors='coerce')
df[JID] = df[JID].astype(str)   # string journal_id for consistent indexing

# Impute controls (within-journal median) to fill small gaps; field/category are
# time-invariant per journal so this only fills the rare journal with an
# entirely-missing value with the sample-wide median.
for c in CONTROLS:
    if c in df.columns:
        df[c] = df.groupby(JID)[c].transform(lambda x: x.fillna(x.median()))
        df[c] = df[c].fillna(df[c].median())

YR_MIN = int(df[YR].min())
df['yr_idx'] = df[YR].astype(int) - YR_MIN

# ─── Journal-level pre-treatment gender-gap baseline (Model A feature only) ──
# Mean gender_gap_y within each journal's own untreated regime (all years for
# never-treated; pre-treatment years only for treated) — a static, per-journal
# proxy for the journal's typical gap level. Constant across the panel, so it
# needs no recursive updating during post-treatment forecasting.
gap_pretreat_mean = df.loc[df['did_treat'] == 0].groupby(JID)[GAP_COL].mean()
df['gap_pretreat_mean'] = df[JID].map(gap_pretreat_mean)
df['gap_pretreat_mean'] = df['gap_pretreat_mean'].fillna(df.loc[df['did_treat'] == 0, GAP_COL].mean())

# ─── Novelty gap (male − female share novel) — replaces avg_novel as a novelty measure ──
# avg_novel_male / avg_novel_female are the journal-year share of novel papers led by
# male / female authors respectively; avg_novel_gap is their difference (male - female).
df['avg_novel_gap'] = df['avg_novel_male'] - df['avg_novel_female']

print(f'Observations  : {len(df):,}')
print(f'Journals      : {df[JID].nunique():,}')
print(f'Year range    : {df[YR].min()} – {df[YR].max()}')

n_never   = (df.groupby(JID)['ever_treated'].max() == 0).sum()
n_treated = (df.groupby(JID)['ever_treated'].max() == 1).sum()
n_untreated_obs = (df['did_treat'] == 0).sum()
print(f'Never-treated journals   : {n_never}')
print(f'Ever-treated  journals   : {n_treated}')
print(f'Untreated-regime obs     : {n_untreated_obs:,} (training sample for all models)')

print('\nOutcome non-missing rates:')
for col in [GAP_COL] + list(NOVELTY_VARS):
    print(f'  {col:20s}: {df[col].notna().mean():.1%}')

In [ ]:
# ─── Descriptive statistics (N, mean, SD, min, max) for all model variables ───
# Computed before training, on (1) the full journal-year panel and (2) the untreated-regime
# training sample (did_treat == 0 & n_papers >= MIN_PAPERS). Controls are shown after the
# within-journal median imputation above, i.e. as they enter the models. Lag features
# (L1..L{N_LAGS}) are shifts of the outcomes, so they are not listed separately.
DESC_VARS = {
    GAP_COL             : 'Gender gap (pp)',
    **NOVELTY_VARS,
    **{c: c for c in CONTROLS},
    'gap_pretreat_mean' : 'Journal pre-treatment mean gender gap',
    YR                  : 'Publication year',
}

def describe_vars(data, var_labels, sample_label):
    rows = []
    for v, lab in var_labels.items():
        s = pd.to_numeric(data[v], errors='coerce').astype(float)
        rows.append({
            'Sample'   : sample_label,
            'Variable' : v,
            'Label'    : lab,
            'N'        : int(s.notna().sum()),
            'Mean'     : round(s.mean(), 4),
            'SD'       : round(s.std(ddof=1), 4),
            'Min'      : round(s.min(), 4),
            'Max'      : round(s.max(), 4),
        })
    return pd.DataFrame(rows)

train_mask = (df['did_treat'] == 0) & (df['n_papers'] >= MIN_PAPERS)
desc_stats = pd.concat([
    describe_vars(df, DESC_VARS, 'Full panel'),
    describe_vars(df.loc[train_mask], DESC_VARS, f'Training sample (did_treat==0, n_papers>={MIN_PAPERS})'),
], ignore_index=True)

print('=== Descriptive statistics ===')
for s, g in desc_stats.groupby('Sample', sort=False):
    print(f'\n--- {s} ---')
    print(g.drop(columns='Sample').to_string(index=False))

out_path = f'{FIG_DIR}/descriptive_stats_{AUTHOR}.csv'
desc_stats.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')

---
## Step 1 — Fit Model A / B / C on the untreated regime

`HistGradientBoostingRegressor` handles `NaN` features natively (no imputation of lag columns
needed — a journal's first `N_LAGS` panel years simply have missing lags, which the model
learns to route through its own missing-value branch). Hyperparameters are tuned with a light
`RandomizedSearchCV` using `GroupKFold` grouped by `journal_id`, so a journal's rows never span
both the train and validation fold (plain K-fold would leak a journal's own lagged outcomes
across the split).

In [ ]:
def fit_gbdt(X, y, groups, cat_mask, label,
             param_distributions=None, n_search_iter=20, cv_splits=4, random_state=42):
    '''Tune + fit a HistGradientBoostingRegressor via grouped RandomizedSearchCV.

    Returns (fitted best_estimator_, best CV R2, best params).
    '''
    if param_distributions is None:
        param_distributions = dict(
            max_iter          = [200, 300, 400, 500],
            max_depth         = [3, 4, 5, 6],
            learning_rate     = [0.02, 0.04, 0.06, 0.1],
            min_samples_leaf  = [10, 20, 30, 50],
            l2_regularization = [0.0, 0.1, 0.5, 1.0],
        )
    if len(y) < 50:
        raise ValueError(f'Too few training obs for {label}: {len(y)}')

    base_model = HistGradientBoostingRegressor(categorical_features=cat_mask, random_state=random_state)
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_search_iter, cv=GroupKFold(n_splits=cv_splits),
        scoring='r2', random_state=random_state, n_jobs=-1,
    )
    search.fit(X, y, groups=groups)
    model = search.best_estimator_   # already refit on all of X, y

    print(f'  {label}: grouped CV R² = {search.best_score_:.3f}  (n_train = {len(y):,})')
    print(f'    best params: {search.best_params_}')
    return model, search.best_score_, search.best_params_


def train_triplet(data, nov_col, gap_col=GAP_COL, controls=CONTROLS,
                   categorical_controls=CATEGORICAL_CONTROLS, n_lags=N_LAGS,
                   jid=JID, yr=YR, did_col='did_treat', min_papers=MIN_PAPERS, **fit_kwargs):
    '''Fit Model A (gender_gap), Model B (novelty, gap-blind), Model C (novelty, gap-aware)
    on the untreated regime (did_treat == 0), for one novelty measure `nov_col`.

    `min_papers`, if set, additionally restricts the *training* rows to journal-years with
    at least that many papers -- gender_gap_y/avg_novel* are noisier averages/ratios in
    low-n_papers journal-years, so this trims the noisiest labels without touching how lags
    are built (those still come from the full, unfiltered panel).
    '''
    gap_lag_cols = [f'{gap_col}_lag{k}' for k in range(1, n_lags + 1)]
    nov_lag_cols = [f'{nov_col}_lag{k}' for k in range(1, n_lags + 1)]
    for k in range(1, n_lags + 1):
        data[f'{gap_col}_lag{k}'] = data.groupby(jid)[gap_col].shift(k)
        data[f'{nov_col}_lag{k}'] = data.groupby(jid)[nov_col].shift(k)

    # Model A excludes novelty entirely — gender gap is assumed not to be influenced
    # by the level of research novelty.
    feat_A = gap_lag_cols + controls + ['yr_idx', 'gap_pretreat_mean']
    feat_B = nov_lag_cols + controls + ['yr_idx']
    feat_C = [gap_col] + gap_lag_cols + nov_lag_cols + controls + ['yr_idx']

    cat_A = [c in categorical_controls for c in feat_A]
    cat_B = [c in categorical_controls for c in feat_B]
    cat_C = [c in categorical_controls for c in feat_C]

    ctrl_mask = data[did_col] == 0
    if min_papers is not None and 'n_papers' in data.columns:
        n_before = ctrl_mask.sum()
        ctrl_mask = ctrl_mask & (data['n_papers'] >= min_papers)
        print(f'  min_papers={min_papers}: training rows {n_before:,} -> {ctrl_mask.sum():,}')

    def get_xy(feat_cols, target):
        m = ctrl_mask & data[target].notna()
        X = data.loc[m, feat_cols].astype(float).values
        y = data.loc[m, target].astype(float).values
        g = data.loc[m, jid].values
        return X, y, g

    print(f'--- Novelty measure: {nov_col} ---')
    XA, yA, gA = get_xy(feat_A, gap_col)
    model_A, r2_A, params_A = fit_gbdt(XA, yA, gA, cat_A, 'Model A  (gender_gap ~ gap lags + controls, no novelty)', **fit_kwargs)

    XB, yB, gB = get_xy(feat_B, nov_col)
    model_B, r2_B, params_B = fit_gbdt(XB, yB, gB, cat_B, f'Model B  ({nov_col} ~ {nov_col} lags only)', **fit_kwargs)

    XC, yC, gC = get_xy(feat_C, nov_col)
    model_C, r2_C, params_C = fit_gbdt(XC, yC, gC, cat_C, f'Model C  ({nov_col} ~ gender_gap + gap lags + {nov_col} lags)', **fit_kwargs)

    return dict(
        model_A=model_A, model_B=model_B, model_C=model_C,
        feat_A=feat_A, feat_B=feat_B, feat_C=feat_C,
        gap_lag_cols=gap_lag_cols, nov_lag_cols=nov_lag_cols,
        r2=dict(A=r2_A, B=r2_B, C=r2_C),
        params=dict(A=params_A, B=params_B, C=params_C),
        n_train=dict(A=len(yA), B=len(yB), C=len(yC)),
    )

print('Model-fitting helpers defined')

---
## Step 2 — Recursive post-treatment counterfactual paths

For each treated journal, walk forward year-by-year through the post-treatment period.
At each step, lag features are pulled from the relevant path's own history dict — which is
seeded with actual pre-treatment values and updated with each new prediction — so later years
never see actual post-treatment outcomes as inputs.

In [ ]:
def get_lags(hist, year_i, n_lags):
    '''Pull L1..Ln lag values from a {year: value} history dict; fill any gaps
    (e.g. missing actual novelty in a lag year) with the nearest available value.'''
    vals = [hist.get(year_i - k, np.nan) for k in range(1, n_lags + 1)]
    if any(pd.isna(v) for v in vals):
        last_ok = next((v for v in vals if not pd.isna(v)), np.nan)
        vals = [v if not pd.isna(v) else last_ok for v in vals]
    return vals


def generate_counterfactuals(data, nov_col, triplet, gap_col=GAP_COL, controls=CONTROLS,
                              n_lags=N_LAGS, jid=JID, yr=YR, yr_min=YR_MIN,
                              treat_col='ever_treated', first_treat_col='first_treat_yr'):
    '''Recursive post-treatment counterfactual forecast for every treated journal,
    for one novelty measure `nov_col`, using its fitted Model A/B/C triplet.

    Returns a journal-year-level DataFrame with actual values, all four predicted
    paths, and the derived deviation/decomposition columns.
    '''
    model_A, model_B, model_C = triplet['model_A'], triplet['model_B'], triplet['model_C']
    records = []

    treated_ids = data.loc[data[treat_col] == 1, jid].unique()
    for jrn in treated_ids:
        jdf = data[data[jid] == jrn].sort_values(yr)
        t0_raw = jdf[first_treat_col].iloc[0]
        if pd.isna(t0_raw):
            continue
        t0 = int(t0_raw)

        pre_df  = jdf[jdf[yr] < t0]
        post_df = jdf[jdf[yr] >= t0]
        if len(pre_df) < n_lags or len(post_df) == 0:
            continue

        # Actual gender-gap history (pre + post) — used for the "actual gap" path.
        gap_hist_actual = {int(r[yr]): r[gap_col] for _, r in jdf.iterrows()}
        # Counterfactual path histories, seeded with actual pre-treatment values.
        gap_hist_0   = {int(r[yr]): r[gap_col] for _, r in pre_df.iterrows()}
        nov_hist_B   = {int(r[yr]): r[nov_col] for _, r in pre_df.iterrows()}
        nov_hist_CG0 = {int(r[yr]): r[nov_col] for _, r in pre_df.iterrows()}
        nov_hist_CGa = {int(r[yr]): r[nov_col] for _, r in pre_df.iterrows()}

        # Pre-treatment: actual == counterfactual for every path, by construction.
        for _, row in pre_df.iterrows():
            records.append(dict(
                journal_id=jrn, year=int(row[yr]), first_treat=t0,
                event_time=int(row[yr]) - t0,
                actual_gender_gap=row[gap_col], pred_gender_gap_0=row[gap_col],
                actual_novelty=row[nov_col], pred_novelty_B_0=row[nov_col],
                pred_novelty_C_G0=row[nov_col], pred_novelty_C_Gactual=row[nov_col],
                post=0,
            ))

        # Post-treatment: iterative forecast, one year at a time.
        for _, row in post_df.iterrows():
            year_i = int(row[yr])
            yr_idx_i = year_i - yr_min
            ctrl_vals = [row.get(c, np.nan) for c in controls]

            # Model A — no-treatment gender-gap path. Novelty is excluded (gender gap is
            # assumed not influenced by the level of research novelty), so only the gap's
            # own lags + controls + gap_pretreat_mean (a static per-journal value, looked
            # up directly since it doesn't vary across the panel) are used.
            gap_lags_0 = get_lags(gap_hist_0, year_i, n_lags)
            xA = np.array(gap_lags_0 + ctrl_vals + [yr_idx_i, row['gap_pretreat_mean']],
                           dtype=float).reshape(1, -1)
            pred_gap0 = float(model_A.predict(xA)[0])
            gap_hist_0[year_i] = pred_gap0

            # Model B — baseline no-treatment novelty path (gap-blind).
            nov_lags_B = get_lags(nov_hist_B, year_i, n_lags)
            xB = np.array(nov_lags_B + ctrl_vals + [yr_idx_i], dtype=float).reshape(1, -1)
            pred_novB0 = float(model_B.predict(xB)[0])
            nov_hist_B[year_i] = pred_novB0

            # Model C, no-treatment gender-gap path (G0): current + lagged gap from the
            # predicted no-treatment path; lagged novelty recursive within this same path.
            nov_lags_CG0 = get_lags(nov_hist_CG0, year_i, n_lags)
            xC_G0 = np.array([pred_gap0] + gap_lags_0 + nov_lags_CG0 + ctrl_vals + [yr_idx_i],
                              dtype=float).reshape(1, -1)
            pred_novCG0 = float(model_C.predict(xC_G0)[0])
            nov_hist_CG0[year_i] = pred_novCG0

            # Model C, actual gender-gap path (Gactual): current + lagged gap from the
            # OBSERVED post-treatment path; lagged novelty still recursive within this path.
            actual_gap = gap_hist_actual.get(year_i, np.nan)
            gap_lags_actual = get_lags(gap_hist_actual, year_i, n_lags)
            nov_lags_CGa = get_lags(nov_hist_CGa, year_i, n_lags)
            xC_Ga = np.array([actual_gap] + gap_lags_actual + nov_lags_CGa + ctrl_vals + [yr_idx_i],
                              dtype=float).reshape(1, -1)
            pred_novCGa = float(model_C.predict(xC_Ga)[0])
            nov_hist_CGa[year_i] = pred_novCGa

            records.append(dict(
                journal_id=jrn, year=year_i, first_treat=t0,
                event_time=year_i - t0,
                actual_gender_gap=actual_gap, pred_gender_gap_0=pred_gap0,
                actual_novelty=row[nov_col], pred_novelty_B_0=pred_novB0,
                pred_novelty_C_G0=pred_novCG0, pred_novelty_C_Gactual=pred_novCGa,
                post=1,
            ))

    result = pd.DataFrame(records)
    if len(result):
        result['gender_gap_deviation'] = result['actual_gender_gap'] - result['pred_gender_gap_0']
        result['total_novelty_deviation'] = result['actual_novelty'] - result['pred_novelty_C_G0']
        result['gap_implied_novelty_component'] = result['pred_novelty_C_Gactual'] - result['pred_novelty_C_G0']
        result['residual_novelty_component'] = result['actual_novelty'] - result['pred_novelty_C_Gactual']
    return result

print('Counterfactual-generation helpers defined')

---
## Step 3 — Run the full pipeline for each novelty measure

In [ ]:
RESULTS = {}
MODELS  = {}

for nov_col, nov_label in NOVELTY_VARS.items():
    triplet = train_triplet(df, nov_col)
    res = generate_counterfactuals(df, nov_col, triplet)
    RESULTS[nov_col] = res
    MODELS[nov_col]  = triplet

    out_path = f'{FIG_DIR}/gbdt3step_{AUTHOR}_{nov_col}.csv'
    res.to_csv(out_path, index=False)
    print(f'  -> {len(res):,} journal-year rows ({res["journal_id"].nunique()} journals) saved to {out_path}\n')

In [ ]:
# ─── Export grouped-CV R² and tuned hyperparameters for all 9 models ────────
MODEL_LABELS = {
    'A': 'gender_gap ~ gap lags + controls (no novelty)',
    'B': 'novelty ~ novelty lags only (gap-blind)',
    'C': 'novelty ~ gender_gap + gap lags + novelty lags (gap-aware)',
}

rows = []
for nov_col, triplet in MODELS.items():
    for m in ['A', 'B', 'C']:
        row = {
            'novelty_var': nov_col,
            'model': m,
            'model_spec': MODEL_LABELS[m],
            'target': GAP_COL if m == 'A' else nov_col,
            'grouped_cv_r2': triplet['r2'][m],
            'n_train': triplet['n_train'][m],
        }
        row.update(triplet['params'][m])
        rows.append(row)

model_summary = pd.DataFrame(rows)
print('=== Grouped-CV R² and tuned hyperparameters (9 models) ===')
print(model_summary.to_string(index=False))

out_path = f'{FIG_DIR}/model_hyperparams_r2.csv'
model_summary.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')

In [ ]:
def summarize_post(res_df, nov_label):
    post = res_df[res_df['post'] == 1]
    return {
        'Novelty measure'               : nov_label,
        'N journals'                    : post['journal_id'].nunique(),
        'N journal-years (post)'        : len(post),
        'Mean gender-gap deviation'     : round(post['gender_gap_deviation'].mean(), 4),
        'Mean total novelty deviation'  : round(post['total_novelty_deviation'].mean(), 4),
        'Mean gap-implied component'    : round(post['gap_implied_novelty_component'].mean(), 4),
        'Mean residual component'       : round(post['residual_novelty_component'].mean(), 4),
    }

summary_tbl = pd.DataFrame([summarize_post(RESULTS[c], l) for c, l in NOVELTY_VARS.items()])
print('=== Post-treatment deviation summary ===')
print(summary_tbl.to_string(index=False))
summary_tbl.to_csv(f'{FIG_DIR}/summary_post_deviation.csv', index=False)
print(f'\nSaved to {FIG_DIR}/summary_post_deviation.csv')

---
## ATT for the gender gap and novelty measures (post-treatment, with 95% CI)

The full journal-year decomposition above (`gender_gap_deviation`, `total_novelty_deviation`,
`gap_implied_novelty_component`, `residual_novelty_component`) is the primary output of this
notebook. It is also useful to collapse each of these deviations into a single **ATT** (average
treatment effect on the treated) with a 95% CI, with journal-clustered inference: average
each treated journal's post-treatment deviation first (one number per journal), then take the
mean and standard error **across journals**, not across journal-years — this avoids treating a
journal with 8 post-treatment years as 8 independent observations when it is really 1 treated
unit observed 8 times.

The combined ATT table below has 7 rows:
1. **Gender gap** — ATT on `gender_gap_deviation` (`actual_gender_gap - pred_gender_gap_0`,
   Model A). This is identical across the three `RESULTS` entries (`avg_novel_gap`,
   `avg_novel_female`, `avg_novel_male`), since Model A never sees novelty and so is fit
   identically regardless of which novelty measure the pipeline is looping over — computed once,
   from whichever entry is convenient.
2. **Total novelty deviation** (one row per novelty measure) — ATT on `total_novelty_deviation`
   (`actual_novelty - pred_novelty_C_G0`): the gap between actual post-treatment novelty and
   what Model C predicts novelty would be given the *predicted* (Model A) no-treatment
   gender-gap path — i.e. the total post-treatment novelty effect, through the gender-gap
   channel and any other treatment-correlated channel Model C's other features pick up.
3. **Gap-implied component** (one row per novelty measure) — ATT on
   `gap_implied_novelty_component` (`pred_novelty_C_Gactual - pred_novelty_C_G0`): the part of
   that novelty deviation attributable specifically to the journal's own observed gender-gap
   change, holding everything else in Model C fixed.

`residual_novelty_component` (`actual_novelty - pred_novelty_C_Gactual`) is left out of the
table below — it is straightforward to add with the same `compute_att` pattern if useful later.

**Caveat:** the CI here reflects cross-journal variability in the observed deviation, not a
formal propagation of Model A's own forecasting uncertainty into Model C — `pred_novelty_C_G0`
is built on Model A's *predicted*, not actual, gender-gap path, but that step's own error
doesn't widen the novelty ATT's SE. This is the same limitation the gender-gap ATT already has
(its SE doesn't reflect GBDT prediction uncertainty either), so the novelty rows are
methodologically consistent with the existing gender-gap row, just extended to the two novelty
quantities above.


In [ ]:
# ─── ATT with 95% CI (journal-clustered): gender gap + novelty measures ─────
from scipy import stats

def compute_att(res_df, dev_col, label, component, novelty_var=None, method='GBDT-3step'):
    '''
    ATT = mean post-treatment deviation, averaged first within journal (one number
    per treated journal), then across journals — journal-clustered SE.
    '''
    post = res_df[res_df['post'] == 1].copy()
    jrn_att = post.groupby('journal_id')[dev_col].mean()
    n      = len(jrn_att)
    att    = jrn_att.mean()
    se     = jrn_att.std(ddof=1) / np.sqrt(n)
    t_stat = att / se if se > 0 else np.nan
    p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1)) if se > 0 else np.nan
    ci_lo  = att - 1.96 * se
    ci_hi  = att + 1.96 * se
    return {
        'Method'      : method,
        'Component'   : component,
        'Novelty var' : novelty_var,
        'Outcome'     : label,
        'N journals'  : n,
        'ATT'         : round(att, 4),
        'SE'          : round(se, 4),
        't-stat'      : round(t_stat, 2),
        'p-value'     : round(p_val, 4),
        'CI_lo_95'    : round(ci_lo, 4),
        'CI_hi_95'    : round(ci_hi, 4),
    }

att_rows = [
    compute_att(next(iter(RESULTS.values())), 'gender_gap_deviation',
                'Gender gap (pp)', component='Gender gap'),
]
for nov_col, nov_label in NOVELTY_VARS.items():
    att_rows.append(compute_att(
        RESULTS[nov_col], 'total_novelty_deviation',
        f'{nov_label} — total deviation', component='Total novelty deviation', novelty_var=nov_col))
    att_rows.append(compute_att(
        RESULTS[nov_col], 'gap_implied_novelty_component',
        f'{nov_label} — gap-implied component', component='Gap-implied component', novelty_var=nov_col))

att_summary = pd.DataFrame(att_rows)

print('=== ATT with 95% CI (journal-clustered): gender gap + novelty measures ===')
print(att_summary.set_index(['Component', 'Outcome']).to_string())

att_summary.to_csv(f'{FIG_DIR}/summary_att_all.csv', index=False)
print(f"\nSaved to {FIG_DIR}/summary_att_all.csv")


In [ ]:
# ─── ATT bar chart with 95% CI whiskers: gender gap + novelty measures ──────
from matplotlib.patches import Patch

COMPONENT_COLORS = {
    'Gender gap'              : '#C44E52',
    'Total novelty deviation' : '#4C72B0',
    'Gap-implied component'   : '#55A868',
}

fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(att_summary) + 1))
y_pos = np.arange(len(att_summary))[::-1]
colors = [COMPONENT_COLORS[c] for c in att_summary['Component']]

ax.barh(y_pos, att_summary['ATT'], xerr=1.96 * att_summary['SE'],
        color=colors, alpha=0.85, height=0.6,
        error_kw={'linewidth': 1.5, 'capsize': 4})
ax.set_yticks(y_pos)
ax.set_yticklabels(att_summary['Outcome'])
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('ATT, 95% CI')
ax.set_title(f'GBDT-3step: ATT summary (AUTHOR={AUTHOR})', fontsize=10, fontweight='bold')
ax.legend(handles=[Patch(color=c, label=l) for l, c in COMPONENT_COLORS.items()],
          fontsize=8, loc='best')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/summary_att_all.pdf', bbox_inches='tight')
plt.show()


---
## Event-time aggregation and figures

Aggregate each of the three journal-year result sets by event time (years relative to first
female EIC appointment), then plot actual vs. predicted gender-gap and novelty trajectories.

In [ ]:
def aggregate_by_event_time(res_df, window=(-6, 6), min_n=5):
    sub = res_df[res_df['event_time'].between(*window)].copy()
    agg_cols = [
        'actual_gender_gap', 'pred_gender_gap_0', 'gender_gap_deviation',
        'actual_novelty', 'pred_novelty_B_0', 'pred_novelty_C_G0', 'pred_novelty_C_Gactual',
        'total_novelty_deviation', 'gap_implied_novelty_component', 'residual_novelty_component',
    ]
    agg = sub.groupby('event_time')[agg_cols].mean().reset_index()
    n = sub.groupby('event_time').size().rename('n').reset_index()
    agg = agg.merge(n, on='event_time')
    agg = agg[agg['n'] >= min_n].reset_index(drop=True)
    return agg


def plot_gender_gap_path(agg, nov_label, save_path=None):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(agg['event_time'], agg['actual_gender_gap'], 'o-', color='#C44E52', lw=2, ms=5, label='Actual')
    ax.plot(agg['event_time'], agg['pred_gender_gap_0'], 's--', color='#4C72B0', lw=2, ms=5,
            label='Predicted (no-treatment, Model A)')
    ax.axvline(0, color='gray', ls=':', lw=1.2)
    ax.set_xlabel('Event time (years relative to first female EIC)')
    ax.set_ylabel('Gender gap (pp)')
    ax.set_title(f'Actual vs. Counterfactual Gender Gap\n(Model A trained with {nov_label} lags)', fontsize=10)
    ax.legend(fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


def plot_novelty_paths(agg, nov_label, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(agg['event_time'], agg['actual_novelty'], 'o-', color='#C44E52', lw=2, ms=5, label='Actual')
    ax.plot(agg['event_time'], agg['pred_novelty_B_0'], 's--', color='#55A868', lw=2, ms=5,
            label='Model B — no-treatment, gap-blind')
    ax.plot(agg['event_time'], agg['pred_novelty_C_G0'], '^--', color='#4C72B0', lw=2, ms=5,
            label='Model C — no-treatment gap path')
    ax.plot(agg['event_time'], agg['pred_novelty_C_Gactual'], 'd--', color='#8172B2', lw=2, ms=5,
            label='Model C — actual gap path')
    ax.axvline(0, color='gray', ls=':', lw=1.2)
    ax.set_xlabel('Event time (years relative to first female EIC)')
    ax.set_ylabel(nov_label)
    ax.set_title(f'Actual vs. Counterfactual Novelty — {nov_label}', fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


def plot_decomposition(agg, nov_label, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(agg['event_time'], agg['total_novelty_deviation'], 'o-', color='black', lw=2, ms=5,
            label='Total novelty deviation')
    ax.plot(agg['event_time'], agg['gap_implied_novelty_component'], 's--', color='#4C72B0', lw=2, ms=5,
            label='Gender-gap-implied component')
    ax.plot(agg['event_time'], agg['residual_novelty_component'], '^--', color='#DD8452', lw=2, ms=5,
            label='Residual component')
    ax.axvline(0, color='gray', ls=':', lw=1.2)
    ax.axhline(0, color='black', ls='-', lw=0.6, alpha=0.4)
    ax.set_xlabel('Event time (years relative to first female EIC)')
    ax.set_ylabel(f'Deviation in {nov_label}')
    ax.set_title(f'Novelty Deviation Decomposition — {nov_label}', fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

print('Aggregation/plotting helpers defined')

In [ ]:
AGGS = {}
for nov_col, nov_label in NOVELTY_VARS.items():
    agg = aggregate_by_event_time(RESULTS[nov_col])
    AGGS[nov_col] = agg
    agg.to_csv(f'{FIG_DIR}/event_time_agg_{nov_col}.csv', index=False)

    plot_gender_gap_path(agg, nov_label, save_path=f'{FIG_DIR}/gap_{nov_col}.pdf')
    plot_novelty_paths(agg, nov_label, save_path=f'{FIG_DIR}/novelty_{nov_col}.pdf')
    plot_decomposition(agg, nov_label, save_path=f'{FIG_DIR}/decomposition_{nov_col}.pdf')

---
## Interpretation guide

### Credibility checks
1. **Grouped CV R²** printed for every one of the 9 models (journal-held-out, not row-held-out)
   — low R² means the corresponding counterfactual path is less trustworthy.
2. **Pre-period trivial fit**: pre-treatment rows have `actual == predicted` by construction
   for all four paths (that's the seed for the recursion), so there is no pre-trend test within
   this design. Rely on the CV R²
   and out-of-sample residuals from Models A/B/C instead.
3. If `total_novelty_deviation` and `gap_implied_novelty_component + residual_novelty_component`
   diverge noticeably, that reflects Model C's own out-of-sample error on the two counterfactual
   paths (G0 vs Gactual), not a bug in the identity — the two paths are separate recursive
   forecasts, not algebraically constrained to sum.

### Using results in the paper
- **Datasets:** `gbdt3step_{AUTHOR}_{novelty_var}.csv` — full journal-year panel (treated
  journals only, pre + post) with all actual/predicted paths and derived deviations.
- **Figures:** `gap_{novelty_var}.pdf`, `novelty_{novelty_var}.pdf`,
  `decomposition_{novelty_var}.pdf` for each of `avg_novel_gap`, `avg_novel_female`, `avg_novel_male`.
- **Table:** `summary_post_deviation.csv` — mean post-treatment deviations by novelty measure.
- **Narrative:** "Under the counterfactual no-treatment gender-gap path, novelty would have
  followed `pred_novelty_C_G0`. The gap between this and the actual post-treatment novelty
  (`total_novelty_deviation`) decomposes into a component consistent with the journal's
  observed gender-gap change (`gap_implied_novelty_component`) and a residual component not
  explained by the gender-gap channel (`residual_novelty_component`).